# Phase 5: Train YOLOv8 Defect Detector in Google Colab (Tesla T4 GPU)

This notebook trains an anchor-free **YOLOv8 Nano** detector on the NEU steel surface defect dataset with 6 defect classes.

### Pipeline Steps:
1. **GPU Hardware Gatekeeper**: Verify Tesla T4 allocation and VRAM availability.
2. **Hugging Face Authentication**: Connect securely via Colab Secrets.
3. **Dataset Verification**: Check partition counts (1,437 train, 180 val, 180 test).
4. **YOLOv8n Model Training**: 50 epochs with TaskAlignedAssigner, CIoU + DFL losses, AdamW optimizer.
5. **Quantitative Evaluation**: Compute mAP@50, mAP@50:95, Precision, Recall on both Val and Test splits.
6. **Visual Inspection Overlays**: Render bounding boxes with confidence scores on test samples.
7. **ONNX Export**: Generate optimized ONNX model for edge deployment.
8. **Hugging Face Hub Upload**: Publish trained weights and telemetry to your Hugging Face repository.

In [ ]:
# 1. Environment & Dependencies
!pip install -q ultralytics huggingface_hub pyyaml matplotlib opencv-python

In [ ]:
# 2. GPU Verification
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected! Please enable T4 GPU in Runtime -> Change runtime type.")

print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# 3. Hugging Face Login
import os
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    api = HfApi()
    print(f"Logged in as: {api.whoami()['name']}")
else:
    print("HF_TOKEN not found. Artifact upload will be skipped.")

In [ ]:
# 4. Run Training Script
!python training/colab/detection/05_train_yolo.py